# FastMCP 服务器
了解核心 FastMCP 服务器类以及如何运行它。

FastMCP 应用程序的核心部分是FastMCP服务器类。该类充当应用程序工具、资源和提示符的主要容器，并管理与 MCP 客户端的通信。

​


## 创建服务器
实例化服务器非常简单。通常需要为服务器提供一个名称，以便在客户端应用程序或日志中识别它。

In [ ]:
from fastmcp import FastMCP

# 创建基本服务器实例
mcp = FastMCP(name="MyAssistantServer")

# 您还可以添加有关如何与服务器交互的说明
mcp_with_instructions = FastMCP(
    name="HelpfulAssistant",
    instructions="""
        This server provides data analysis tools.
        Call get_average() to analyze numerical data.
        """
)


构造FastMCP函数接受几个参数：

- `name`：（可选）服务器的可读名称。默认为“FastMCP”。
- `instructions`：（可选）有关如何与此服务器交互的说明。这些说明可帮助客户端了解服务器的用途和可用功能。
- `lifespan`：（可选）用于服务器启动和关闭逻辑的异步上下文管理器函数。
- `tags`：（可选）一组用于标记服务器本身的字符串。
- `**settings`：对应于附加ServerSettings配置的关键字参数
​


## 成分
FastMCP 服务器向客户端公开几种类型的组件：

### 工具
工具是客户端可以调用来执行操作或访问外部系统的功能。

In [ ]:
@mcp.tool()
def multiply(a: float, b: float) -> float:
    """Multiplies two numbers together."""
    return a * b

### 资源
资源公开客户端可以读取的数据源。

In [ ]:
@mcp.resource("data://config")
def get_config() -> dict:
    """Provides the application configuration."""
    return {"theme": "dark", "version": "1.0"}

### 资源模板
资源模板是允许客户端请求特定数据的参数化资源。

In [ ]:
@mcp.resource("users://{user_id}/profile")
def get_user_profile(user_id: int) -> dict:
    """Retrieves a user's profile by ID."""
    # The {user_id} in the URI is extracted and passed to this function
    return {"id": user_id, "name": f"User {user_id}", "status": "active"}

### 提示
提示是用于指导 LLM 的可重复使用的消息模板。

In [ ]:
@mcp.prompt()
def analyze_data(data_points: list[float]) -> str:
    """Creates a prompt asking for analysis of numerical data."""
    formatted_data = ", ".join(str(point) for point in data_points)
    return f"Please analyze these data points: {formatted_data}"

## 运行服务器

FastMCP 服务器需要一种传输机制来与客户端通信。启动服务器的方法通常是在 FastMCP 实例上调用 mcp.run() 方法，通常是在服务器主脚本的 `if __name__ == "__main__"`: 代码块中。这种模式确保了与各种 MCP 客户端的兼容性。

In [ ]:
# my_server.py
from fastmcp import FastMCP

mcp = FastMCP(name="MyServer")

@mcp.tool()
def greet(name: str) -> str:
    """Greet a user by name."""
    return f"Hello, {name}!"

if __name__ == "__main__":
    # This runs the server, defaulting to STDIO transport
    mcp.run()
    
    # To use a different transport, e.g., HTTP:
    # mcp.run(transport="streamable-http", host="127.0.0.1", port=9000)

FastMCP 支持多种传输选项：

- STDIO（默认，用于本地工具）
- 可流式传输的 HTTP（推荐用于 Web 服务）
- SSE（旧版 Web 传输，已弃用）

该服务器也可以使用 FastMCP CLI 运行。

## 组合服务器
FastMCP 支持使用`import_server`静态复制和`mount`实时链接两种方式将多个服务器组合在一起。这允许您将大型应用程序组织成模块化组件或重用现有服务器。

In [ ]:
# Example: Importing a subserver
from fastmcp import FastMCP
import asyncio

main = FastMCP(name="Main")
sub = FastMCP(name="Sub")

@sub.tool()
def hello(): 
    return "hi"

# Mount directly
main.mount("sub", sub)

## Proxying Servers

使用 `FastMCP.as_proxy`，FastMCP 可充当任何 MCP 服务器（本地或远程）的代理，让您可以桥接传输或为现有服务器添加前端。例如，您可以通过 stdio 在本地公开远程 SSE 服务器，反之亦然。

In [ ]:
from fastmcp import FastMCP, Client

backend = Client("http://example.com/mcp/sse")
proxy = FastMCP.as_proxy(backend, name="ProxyServer")
# Now use the proxy like any FastMCP server

## 服务器配置
服务器行为，例如传输设置（主机、SSE 的端口）以及如何处理重复组件，可以通过 `ServerSettings`进行配置。这些设置可以在`FastMCP`初始化期间传递，通过环境变量（以`FASTMCP_SERVER_` 为前缀）设置，或从文件`.env`加载。

关键配置选项
- `host`：SSE 传输的主机地址（默认值：“127.0.0.1”）
- `port`：SSE 传输的端口号（默认值：8000）
- `log_level`：日志记录级别（默认：“INFO”）
- `on_duplicate_tools`：如何处理重复的工具注册
- `on_duplicate_resources`：如何处理重复的资源注册
- `on_duplicate_prompts`：如何处理重复的提示注册
所有这些都可以在创建FastMCP实例时直接作为参数进行配置。

In [ ]:
from fastmcp import FastMCP

# Configure during initialization
mcp = FastMCP(
    name="ConfiguredServer",
    port=8080, # Directly maps to ServerSettings
    on_duplicate_tools="error" # Set duplicate handling
)

# Settings are accessible via mcp.settings
print(mcp.settings.port) # Output: 8080
print(mcp.settings.on_duplicate_tools) # Output: "error"

## 自定义工具序列化
默认情况下，当工具返回值需要转换为文本时，FastMCP 会将其序列化为 JSON。你可以在创建服务器时提供一个`tool_serializer`函数来自定义此行为：

In [ ]:
import yaml
from fastmcp import FastMCP

# Define a custom serializer that formats dictionaries as YAML
def yaml_serializer(data):
    return yaml.dump(data, sort_keys=False)

# Create a server with the custom serializer
mcp = FastMCP(name="MyServer", tool_serializer=yaml_serializer)

@mcp.tool()
def get_config():
    """Returns configuration in YAML format."""
    return {"api_key": "abc123", "debug": True, "rate_limit": 100}

序列化器函数接受任何数据对象并返回字符串表示形式。此函数适用于工具**返回的所有非字符串值**。**已返回字符串的工具将绕过序列化器**。

当您想要执行以下操作时，此自定义非常有用：

- 以特定方式格式化数据（如 YAML 或自定义格式）
- 控制特定的序列化选项（如缩进或排序）
- 在将数据发送给客户端之前添加元数据或转换数据

## 验证

FastMCP 支持 OAuth 2.0 身份验证，允许服务器保护其工具和资源。在 FastMCP 初始化过程中，可通过提供 `auth_server_provider` 和 `auth` 设置来进行配置。


In [ ]:
from fastmcp import FastMCP
from mcp.server.auth.settings import AuthSettings #, ... other auth imports
# from your_auth_implementation import MyOAuthServerProvider # Placeholder

# Create a server with authentication (conceptual example)
# mcp = FastMCP(
#     name="SecureApp",
#     auth_server_provider=MyOAuthServerProvider(),
#     auth=AuthSettings(
#         issuer_url="https://myapp.com",
#         # ... other OAuth settings ...
#         required_scopes=["myscope"],
#     ),
# )

由于当前 MCP SDK 的授权提供程序接口级别较低，因此详细的实现方法超出了快速示例的范围。有关实现 `OAuthAuthorizationServerProvider`的具体细节，请参阅 MCP SDK 文档。FastMCP 通过将提供程序和设置传递给底层 MCP 服务器与之集成。